In [6]:
from pathlib import Path
import subprocess

from PIL import Image, ImageDraw, ImageFont
from praatio import textgrid


In [15]:
PROJECT_ROOT = Path("/workspace/storyfx")

EXCERPT_ID = "excerpt_001"

TEXTGRID_PATH = (
    PROJECT_ROOT
    / "data"
    / EXCERPT_ID
    / "alignment"
    / f"{EXCERPT_ID}_fa.TextGrid"
)


AUDIO_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "augmented_stories"
    / f"{EXCERPT_ID}_augmented.wav"
)

OUTPUT_VIDEO = (
    PROJECT_ROOT
    / "outputs"
    / "augmented_stories"
    / f"{EXCERPT_ID}_augmented_karaoke.mkv"
)

In [17]:
WIDTH = 1280
HEIGHT = 720

FPS = 24

BACKGROUND = (15, 15, 18)

TEXT_COLOR = (235, 235, 235)

HIGHLIGHT_BACKGROUND = (245, 200, 60)
HIGHLIGHT_TEXT = (20, 20, 20)

FONT_SIZE = 48

FONT_PATH = (
    "/usr/share/fonts/truetype/dejavu/"
    "DejaVuSans.ttf"
)

MAX_TEXT_WIDTH = 1050

LINE_GAP = 20

WORD_SPACING = 16

HIGHLIGHT_PAD_X = 8
HIGHLIGHT_PAD_Y = 4
HIGHLIGHT_RADIUS = 8


# Caption grouping
MAX_WORDS_PER_BLOCK = 28
MAX_BLOCK_DURATION = 10.0

# Large pause = new block.
BLOCK_GAP_S = 1.0

# Keep block visible briefly after its final word.
BLOCK_HOLD_S = 0.35


# ============================================================
# LOAD MFA WORDS
# ============================================================

tg = textgrid.openTextgrid(
    str(TEXTGRID_PATH),
    includeEmptyIntervals=True,
)

word_tier = tg.getTier(
    "words"
)

words = []

for start, end, text in word_tier.entries:

    text = text.strip()

    if not text:
        continue

    words.append({
        "start": float(start),
        "end": float(end),
        "text": text,
    })


print(
    f"Loaded {len(words)} words"
)


# ============================================================
# GROUP WORDS INTO TEXT BLOCKS
# ============================================================

def make_blocks(words):

    blocks = []

    current = []

    for word in words:

        if not current:
            current.append(word)
            continue

        block_duration = (
            word["end"]
            - current[0]["start"]
        )

        gap = (
            word["start"]
            - current[-1]["end"]
        )

        should_split = (
            len(current)
            >= MAX_WORDS_PER_BLOCK
            or block_duration
            > MAX_BLOCK_DURATION
            or gap
            >= BLOCK_GAP_S
        )

        if should_split:

            blocks.append(
                current
            )

            current = []

        current.append(
            word
        )

    if current:
        blocks.append(
            current
        )

    return blocks


blocks = make_blocks(
    words
)

print(
    f"Created {len(blocks)} text blocks"
)


# ============================================================
# FONT / TEXT MEASUREMENT
# ============================================================

font = ImageFont.truetype(
    FONT_PATH,
    FONT_SIZE,
)

measurement_image = Image.new(
    "RGB",
    (WIDTH, HEIGHT),
)

measurement_draw = ImageDraw.Draw(
    measurement_image
)


def measure_word(text):

    bbox = measurement_draw.textbbox(
        (0, 0),
        text,
        font=font,
    )

    width = (
        bbox[2] - bbox[0]
    )

    height = (
        bbox[3] - bbox[1]
    )

    return width, height


space_width = WORD_SPACING


# ============================================================
# PRECOMPUTE BLOCK LAYOUT
# ============================================================

def layout_block(block):

    lines = []
    current_line = []
    current_width = 0

    # ----------------------------------------
    # Word wrapping
    # ----------------------------------------

    for index, word in enumerate(block):

        word_width, word_height = (
            measure_word(
                word["text"]
            )
        )

        additional_width = (
            word_width
            if not current_line
            else space_width + word_width
        )

        if (
            current_line
            and current_width
            + additional_width
            > MAX_TEXT_WIDTH
        ):

            lines.append(
                current_line
            )

            current_line = []
            current_width = 0

        current_line.append({
            "index": index,
            "width": word_width,
            "height": word_height,
        })

        if current_width == 0:
            current_width = word_width
        else:
            current_width += (
                space_width
                + word_width
            )

    if current_line:
        lines.append(
            current_line
        )


    # ----------------------------------------
    # Vertical layout
    # ----------------------------------------

    font_bbox = measurement_draw.textbbox(
        (0, 0),
        "Ag",
        font=font,
    )

    line_height = (
        font_bbox[3]
        - font_bbox[1]
    )

    total_height = (
        len(lines)
        * line_height
        +
        max(
            0,
            len(lines) - 1
        )
        * LINE_GAP
    )

    y = (
        HEIGHT // 2
        - total_height // 2
    )

    positions = {}


    # ----------------------------------------
    # Horizontal centering
    # ----------------------------------------

    for line in lines:

        line_width = sum(
            item["width"]
            for item in line
        )

        line_width += (
            space_width
            * max(
                0,
                len(line) - 1
            )
        )

        x = (
            WIDTH // 2
            - line_width // 2
        )

        for item in line:

            index = item["index"]

            positions[index] = {
                "x": x,
                "y": y,
                "width": item["width"],
                "height": line_height,
            }

            x += (
                item["width"]
                + space_width
            )

        y += (
            line_height
            + LINE_GAP
        )


    return positions


block_layouts = [
    layout_block(block)
    for block in blocks
]


# ============================================================
# AUDIO DURATION
# ============================================================

import torchaudio

info = torchaudio.info(
    str(AUDIO_PATH)
)

audio_duration = (
    info.num_frames
    / info.sample_rate
)

print(
    f"Audio:"
    f" {info.sample_rate} Hz"
)

print(
    f"Duration:"
    f" {audio_duration:.2f}s"
)


# ============================================================
# FRAME RENDERER
# ============================================================

def draw_block(
    frame,
    block,
    layout,
    current_time,
):

    draw = ImageDraw.Draw(
        frame
    )

    active_index = None


    # Determine active word.
    for i, word in enumerate(block):

        if (
            word["start"]
            <= current_time
            <= word["end"]
        ):

            active_index = i
            break


    # ----------------------------------------
    # Draw highlight background first
    # ----------------------------------------

    if active_index is not None:

        pos = layout[
            active_index
        ]

        word = block[
            active_index
        ]

        bbox = draw.textbbox(
            (
                pos["x"],
                pos["y"],
            ),
            word["text"],
            font=font,
        )

        rect = (
            bbox[0]
            - HIGHLIGHT_PAD_X,
            bbox[1]
            - HIGHLIGHT_PAD_Y,
            bbox[2]
            + HIGHLIGHT_PAD_X,
            bbox[3]
            + HIGHLIGHT_PAD_Y,
        )

        draw.rounded_rectangle(
            rect,
            radius=HIGHLIGHT_RADIUS,
            fill=HIGHLIGHT_BACKGROUND,
        )


    # ----------------------------------------
    # Draw words
    # ----------------------------------------

    for i, word in enumerate(block):

        pos = layout[i]

        color = (
            HIGHLIGHT_TEXT
            if i == active_index
            else TEXT_COLOR
        )

        draw.text(
            (
                pos["x"],
                pos["y"],
            ),
            word["text"],
            font=font,
            fill=color,
        )


# ============================================================
# FFMPEG
# ============================================================

OUTPUT_VIDEO.parent.mkdir(
    parents=True,
    exist_ok=True,
)

command = [
    "ffmpeg",
    "-y",

    # Raw Pillow frames.
    "-f",
    "rawvideo",

    "-vcodec",
    "rawvideo",

    "-pix_fmt",
    "rgb24",

    "-s",
    f"{WIDTH}x{HEIGHT}",

    "-r",
    str(FPS),

    "-i",
    "-",

    # Original audio.
    "-i",
    str(AUDIO_PATH),

    # Explicit stream mapping.
    "-map",
    "0:v:0",

    "-map",
    "1:a:0",

    # Video encoding only.
    "-c:v",
    "libx264",

    "-preset",
    "medium",

    "-crf",
    "18",

    "-pix_fmt",
    "yuv420p",

    # IMPORTANT:
    # Keep audio stream unchanged.
    "-c:a",
    "copy",

    "-shortest",

    str(OUTPUT_VIDEO),
]


print(
    "\nRendering video..."
)

process = subprocess.Popen(
    command,
    stdin=subprocess.PIPE,
)


# ============================================================
# RENDER FRAMES
# ============================================================

total_frames = int(
    audio_duration
    * FPS
)


block_index = 0


for frame_number in range(
    total_frames
):

    current_time = (
        frame_number
        / FPS
    )


    # Advance block when necessary.
    while (
        block_index
        < len(blocks) - 1
        and current_time
        > blocks[
            block_index
        ][-1]["end"]
        + BLOCK_HOLD_S
    ):

        block_index += 1


    block = blocks[
        block_index
    ]

    block_start = (
        block[0]["start"]
    )

    block_end = (
        block[-1]["end"]
        + BLOCK_HOLD_S
    )


    frame = Image.new(
        "RGB",
        (
            WIDTH,
            HEIGHT,
        ),
        BACKGROUND,
    )


    if (
        block_start
        <= current_time
        <= block_end
    ):

        draw_block(
            frame,
            block,
            block_layouts[
                block_index
            ],
            current_time,
        )


    process.stdin.write(
        frame.tobytes()
    )


    if (
        frame_number
        % (FPS * 10)
        == 0
    ):

        print(
            f"{current_time:.1f}"
            f" / "
            f"{audio_duration:.1f}s"
        )


# ============================================================
# FINISH
# ============================================================

process.stdin.close()

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"FFmpeg failed with "
        f"code {return_code}"
    )


print(
    "\nSaved karaoke video to:"
)

print(
    OUTPUT_VIDEO
)

Loaded 971 words
Created 40 text blocks
Audio: 44100 Hz
Duration: 388.60s

Rendering video...
0.0 / 388.6s


/tmp/ipykernel_38559/2923405993.py:338: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  info = torchaudio.info(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:20: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  s = torchaudio.io.Str

BrokenPipeError: [Errno 32] Broken pipe